In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import os
from datetime import datetime
from tensorflow import keras 

In [ ]:
df=pd.read_csv(r"C:\Users\VICTUS\.keras\datasets\jena_climate_2009_2016_extracted\jena_climate_2009_2016.csv")
df


In [ ]:
df.index=pd.to_datetime(df['Date Time'],format='%d.%m.%Y %H:%M:%S' )

In [ ]:
print(df.describe())
df.shape
print(df.info())


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df['Date Time'][:3000],df['T (degC)'][:3000],label="temp over time",color='red')
plt.show()

In [ ]:
df[['T (degC)']].boxplot(figsize=(12,6)) #to find outliers in the dataset the dots above and below 33 and -15 respectively
#did not remove the outlier as the data is of temprature 

In [ ]:
numeric_data=df.select_dtypes(include=["float64","int64"])
plt.figure(figsize=(12,6))
sns.heatmap(numeric_data.corr(),cmap='coolwarm',annot=True)
plt.title("relation between data")
plt.show()

In [ ]:
#prepare for  the Lstm model
df_numeric = df.drop(columns=['Date Time'])
df_numeric=df_numeric[5::6]
temprature =df_numeric["T (degC)"]

df_numeric.head()





In [ ]:
print(df_numeric.dtypes)
print(df_numeric.isna())
print(df_numeric.shape)

In [ ]:

training_data_len=int(np.ceil(len(df_numeric)*0.95))
training_data=df_numeric[:training_data_len]
scaler=StandardScaler()
scaled_data_tr=scaler.fit_transform(df_numeric[:training_data_len])



In [ ]:
x_train,y_train=[],[]
window_size=24
for i in range(window_size,len(training_data)):
    x_train.append(scaled_data_tr[i-window_size:i,1])
    y_train.append(scaled_data_tr[i,1])

x_train=np.array(x_train)
y_train=np.array(y_train) 

x_train=np.reshape(x_train,(x_train.shape[0],x_train.shape[1],1))

In [ ]:
#building the model
model=keras.models.Sequential()

#first layer 
model.add(keras.layers.LSTM(64,return_sequences=True,input_shape=(x_train.shape[1],1)))
          

#second layer 
model.add(keras.layers.LSTM(64,return_sequences=False))

#third layer (DENSE LAYER)
model.add(keras.layers.Dense(128,activation="relu"))

#fourth layer (dropout regularization layer)
model.add(keras.layers.Dropout(0.5))

#output layer 
model.add(keras.layers.Dense(1))

model.summary()
model.compile(optimizer="adam",
                loss="mae",
                metrics=[keras.metrics.RootMeanSquaredError()])




In [ ]:
training =model.fit(x_train,y_train,epochs=20,batch_size=32)





In [ ]:
test_data=scaler.transform(df_numeric[training_data_len-24:])

x_test,y_test=[],df_numeric[training_data_len:]

for i in range(24,len(test_data)):
    x_test.append(test_data[i-24:i,1])

x_test=np.array(x_test)
x_test=np.reshape(x_test,(x_test.shape[0],x_test.shape[1],1))
predictions=model.predict(x_test)

dummy_matrix = np.zeros((len(predictions), test_data.shape[1]))

dummy_matrix[:, 1] = predictions.flatten()

unscaled_matrix = scaler.inverse_transform(dummy_matrix)

final_predictions = unscaled_matrix[:, 1]

In [ ]:
#plotting
train=df_numeric[:training_data_len]
test=df_numeric[training_data_len:]


test=test.copy()
test['predictions']=predictions
plt.figure(figsize=(12,6))
plt.plot(train.index,train['T (degC)'],label="train (ACTUAL)",color='blue')
plt.plot(test.index,test['T (degC)'],label="test (ACTUAL)",color='red')
plt.plot(test.index,test['predictions'],label="prediction",color='yellow')
plt.xlabel("date")
plt.ylabel("temprature")
plt.legend()
plt.show()